In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Mxfp4Config
import torch

# Only reasoning models
# del model
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# model_id = "openai/gpt-oss-20b" # Ends reasoning with <|channel|>final<|message|> sequence, ends output message with <|return|>

# model_id = "google/gemma-4-26B-A4B-it" # Ends reasoning with <channel|>, ends output message with <turn|>

model_id = "Qwen/Qwen3.6-35B-A3B"

# quantization_config = Mxfp4Config(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype="float16",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
# )

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # match Gemma's native dtype, not float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    # quantization_config=bnb_config, 
    device_map="auto",
    torch_dtype="auto",
    # device_map={"": 0}
    )

ImportError: /home/stulcrad/.local/lib/python3.12/site-packages/torch/lib/libtorch_cuda.so: undefined symbol: ncclCommResume

In [ ]:
from utils.model_reasoning_utils import extract_harmony_final_channel

message = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi, how are you doing?"},
]

prompt = tokenizer.apply_chat_template(
    message,
    tokenize=True,
    reasoning_effort='low',
    add_generation_prompt=True,
    return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(**prompt, max_new_tokens=2000)

In [5]:
print(tokenizer.decode(prompt['input_ids'][0], skip_special_tokens=False))
print(tokenizer.decode(output[0][prompt['input_ids'].shape[1]:], skip_special_tokens=False))
# print(extract_harmony_final_channel(response))

Failed to reload module 'torch.torch_version' from file '/home/stulcrad/.local/lib/python3.12/site-packages/torch/torch_version.py'
Traceback (most recent call last):
  File "/home/stulcrad/.local/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/home/stulcrad/.local/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 625, in superreload
    update_generic(old_obj, new_obj)
  File "/home/stulcrad/.local/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 451, in update_generic
    update(a, b)
  File "/home/stulcrad/.local/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 419, in update_class
    update_instances(old, new)
  File "/home/stulcrad/.local/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 377, in update_instances
    object.__setattr__(ref, "__class__", new)
TypeError: __class__ assignment: 'TorchVersion' object layo

: 

: 

: 

In [11]:
for token, token_id in zip(tokenizer.convert_ids_to_tokens(output[0][prompt['input_ids'].shape[1]:]), output[0][prompt['input_ids'].shape[1]:]):
    print(f"{token}: {token_id}")

<|channel>: 100
thought: 45518

: 107
": 236775
Can: 8574
▁you: 611
▁think: 1751
?": 7462

: 107
The: 818
▁user: 2430
▁is: 563
▁asking: 10980
▁about: 1003
▁the: 506
▁nature: 4135
▁of: 529
▁my: 1041
▁cognition: 87394
,: 236764
▁consciousness: 28625
,: 236764
▁or: 653
▁intelligence: 14020
.: 236761
▁They: 2195
▁want: 1461
▁to: 531
▁know: 1281
▁if: 768
▁my: 1041
▁processing: 8487
▁resembles: 52235
▁human: 3246
▁thought: 3305
.: 236761


: 108
▁▁▁▁: 140
*: 236829
▁▁▁: 139
*: 236829
What: 3689
▁am: 1006
▁I: 564
?: 236881
*: 236829
▁A: 562
▁large: 2455
▁language: 5192
▁model: 2028
▁(: 568
LL: 2182
M: 236792
).: 769

: 107
▁▁▁▁: 140
*: 236829
▁▁▁: 139
*: 236829
How: 3910
▁do: 776
▁I: 564
▁work: 981
?: 236881
*: 236829
▁I: 564
▁predict: 4773
▁the: 506
▁next: 2148
▁token: 8369
▁in: 528
▁a: 496
▁sequence: 7501
▁based: 2721
▁on: 580
▁patterns: 9935
▁learned: 8683
▁from: 699
▁massive: 12566
▁datasets: 32542
.: 236761
▁I: 564
▁use: 1161
▁complex: 3996
▁mathematical: 23093
▁weights: 18710
▁(: 568
ne

In [ ]:
from utils.TokTrie import TokTrie, build_toktrie_from_tokenizer
from utils.TrieSpanConstrainedProcessorTokenAware import TrieSpanConstrainedProcessorTokenAware
from utils.model_reasoning_utils import reasoning_ended
from utils.system_prompts import SYSTEM_PROMPT_CONSTR_GEN

toktrie = build_toktrie_from_tokenizer(tokenizer)

labels = ["PER", "LOC", "ORG", "MISC"]

input_text = "Radek was born in Prague. He has a dog named Rex."

reasoning_model=True

gpt_marker = "<|channel|>final<|message|>"
gpt_marker_ids = tokenizer(gpt_marker, add_special_tokens=False).input_ids

processor = TrieSpanConstrainedProcessorTokenAware(
    labels, input_text, tokenizer, toktrie, reasoning_model, reasoning_ended,
    gpt_marker
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT_CONSTR_GEN},
    {"role": "user", "content": input_text},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    reasoning_effort='low',
    add_generation_prompt=True,
    return_tensors="pt",
    enable_thinking=True,
).to(model.device)

